# Big coding · KV cache from scratch

**Goal:** implement cached causal self-attention and prove that incremental decoding matches full-sequence attention.

**Prerequisites:** PyTorch tensor shapes, matrix multiplication, softmax, and causal attention. **Suggested time:** 60–90 minutes. **Runtime:** Python 3 with PyTorch; CPU is sufficient. No external data or pretrained weights.

Save a copy in Drive before editing. Run cells in order. The starter intentionally raises `NotImplementedError`; finish it before running its tests. The reference solution comes after the tests.

## 1. Problem and contract

Implement `cached_attention(x_new, cache=None)` for one attention layer with fixed projection matrices. The input can be an entire prompt (prefill), a single decode token, or a chunk of new tokens.

- `x_new`: `[B, T_new, D_model]`, with `T_new >= 1`.
- `Wq`, `Wk`, `Wv`, `Wo`: `[D_model, D_model]`; `H` heads, `Dh = D_model // H`.
- Cache: `None` or `(K, V)`, each `[B, H, T_past, Dh]`.
- Return `(output, updated_cache)` with output `[B, T_new, D_model]` and cache length `T_past + T_new`.
- Project only new tokens. Append their keys and values along the sequence axis. Each new query may see all previous tokens and itself, but no later token in the new chunk.
- Use `1 / sqrt(Dh)` scaling. No padding, dropout, positional encoding, or autograd is needed in this exercise. Treat caches as inference-only tensors; batch size, device, and dtype remain fixed.

**Deliverables:** implementation, passing tests, and a short explanation of why KV caching avoids repeated projections but still reads the history.


In [ ]:
import math
import time
import statistics
import torch

torch.manual_seed(7)
torch.set_num_threads(1)
B, D, H = 2, 32, 4
DH = D // H
Wq, Wk, Wv, Wo = [torch.randn(D, D) / math.sqrt(D) for _ in range(4)]

def split_heads(x):
    b, t, _ = x.shape
    return x.reshape(b, t, H, DH).transpose(1, 2)

def merge_heads(x):
    b, _, t, _ = x.shape
    return x.transpose(1, 2).contiguous().reshape(b, t, D)

@torch.no_grad()
def full_attention(x):
    q, k, v = [split_heads(x @ w) for w in (Wq, Wk, Wv)]
    t = x.shape[1]
    future = torch.ones(t, t, dtype=torch.bool, device=x.device).triu(1)
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(DH)
    weights = scores.masked_fill(future, -torch.inf).softmax(dim=-1)
    return merge_heads(weights @ v) @ Wo

x = torch.randn(B, 17, D)
print('PyTorch:', torch.__version__, '| input:', tuple(x.shape))


## 2. Your implementation

1. Project and split only `x_new` into heads.
2. Read the previous cache length, then append new keys and values.
3. Build query positions starting at `T_past` and key positions starting at zero.
4. Mask keys where `key_position > query_position` before softmax.
5. Merge heads and apply `Wo`; return the extended cache.

**Common trap:** an ordinary `T_new × T_total` upper-triangular mask with no past-length offset is wrong after prefill. A one-token decode query should see the entire existing cache plus itself.


In [ ]:
@torch.no_grad()
def cached_attention(x_new, cache=None):
    # TODO: implement steps 1–5. Do not recompute projections of past inputs.
    raise NotImplementedError('Implement cached_attention, then run the tests below.')


## 3. Correctness tests

The same test harness works for your implementation and the reference. It tests prefill, token-by-token decode, irregular chunks, first-token behavior, cache growth, preservation of earlier cache entries, and future-token isolation. Numerical equality uses tolerances because matrix operation ordering can change rounding.


In [ ]:
def check_implementation(fn):
    expected = full_attention(x)
    for sizes in ([17], [1] * 17, [5, 1, 4, 7]):
        cache = None
        outputs = []
        offset = 0
        for size in sizes:
            previous = None if cache is None else tuple(t.clone() for t in cache)
            out, cache = fn(x[:, offset:offset + size], cache)
            offset += size
            assert out.shape == (B, size, D)
            assert all(t.shape == (B, H, offset, DH) for t in cache)
            assert all(torch.isfinite(t).all() for t in (out, *cache))
            if previous is not None:
                for old, new in zip(previous, cache):
                    torch.testing.assert_close(new[:, :, :offset-size], old)
            outputs.append(out)
        torch.testing.assert_close(torch.cat(outputs, dim=1), expected, atol=2e-5, rtol=2e-5)
    # A single visible key has probability one.
    first, _ = fn(x[:, :1])
    torch.testing.assert_close(first, (x[:, :1] @ Wv) @ Wo, atol=2e-5, rtol=2e-5)
    # Altering future inputs cannot change earlier outputs, including within a chunk.
    altered = x.clone()
    altered[:, 8:] += 100
    before, _ = fn(x)
    after, _ = fn(altered)
    torch.testing.assert_close(before[:, :8], after[:, :8], atol=2e-5, rtol=2e-5)
    print('PASS: prefill, incremental decode, chunked decode, cache preservation, causality.')

check_implementation(cached_attention)


## 4. Reference solution — open after attempting the exercise

Run this cell to define a separate reference function. It does not overwrite your implementation. Notice how absolute query positions incorporate the cache length.


In [ ]:
@torch.no_grad()
def reference_cached_attention(x_new, cache=None):
    q, new_k, new_v = [split_heads(x_new @ w) for w in (Wq, Wk, Wv)]
    past = 0 if cache is None else cache[0].shape[2]
    if cache is None:
        k, v = new_k, new_v
    else:
        k = torch.cat((cache[0], new_k), dim=2)
        v = torch.cat((cache[1], new_v), dim=2)
    query_positions = past + torch.arange(x_new.shape[1], device=x_new.device)
    key_positions = torch.arange(k.shape[2], device=x_new.device)
    future = key_positions[None, :] > query_positions[:, None]
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(DH)
    weights = scores.masked_fill(future, -torch.inf).softmax(dim=-1)
    out = merge_heads(weights @ v) @ Wo
    return out, (k, v)

check_implementation(reference_cached_attention)


## 5. Timing experiment

Compare two ways to process the same stream of new input tokens: recompute full attention on every growing prefix, or process only each new token with a cache. These inputs are fixed random vectors, not sampled model outputs. The experiment measures this attention layer, not end-to-end LLM generation.

This tiny CPU benchmark includes Python overhead and repeated `torch.cat` allocations. It is illustrative, not a speed guarantee. Production caches often preallocate memory. A new query still reads all cached keys/values; memory grows with sequence length. For this layer, cache storage is `2 × B × H × T × Dh × bytes_per_element`.


In [ ]:
@torch.no_grad()
def uncached_stream(tokens):
    return torch.cat([full_attention(tokens[:, :i+1])[:, -1:] for i in range(tokens.shape[1])], dim=1)

@torch.no_grad()
def cached_stream(tokens):
    cache = None
    outputs = []
    for i in range(tokens.shape[1]):
        out, cache = reference_cached_attention(tokens[:, i:i+1], cache)
        outputs.append(out)
    return torch.cat(outputs, dim=1)

def median_seconds(fn, tokens, repeats=5):
    fn(tokens)  # warm-up
    samples = []
    for _ in range(repeats):
        start = time.perf_counter()
        fn(tokens)
        samples.append(time.perf_counter() - start)
    return statistics.median(samples)

for length in (32, 64, 128):
    tokens = torch.randn(B, length, D)
    torch.testing.assert_close(cached_stream(tokens), uncached_stream(tokens), atol=2e-5, rtol=2e-5)
    slow = median_seconds(uncached_stream, tokens)
    fast = median_seconds(cached_stream, tokens)
    cache_bytes = 2 * B * H * length * DH * tokens.element_size()
    print(f'T={length:3d} | uncached={slow:.4f}s cached={fast:.4f}s ratio={slow/fast:.2f}x | cache={cache_bytes:,} bytes')


## 6. Review questions and extensions

1. Why is no additional mask needed for single-token decoding when the cache contains only past and current tokens?
2. Which projections are reused? Why does attention cost per new token still grow with context length?
3. Why must a cache belong to the same sequence, layer, model weights, and positional convention?
4. Extend the implementation to a preallocated cache; measure how avoiding `torch.cat` changes latency.
5. Add RoPE with absolute positions. Which keys should be stored, and why must decode positions start at the cached length?
6. How would grouped-query attention change cache shape and memory accounting?

**Scope:** a single inference-only attention layer. A full decoder additionally needs per-layer caches, positional handling, residual blocks, MLPs, normalization, padding, and generation logic.

Reference API: [PyTorch tensor operations](https://docs.pytorch.org/docs/stable/torch.html).
